In [1]:
import pandas as pd
import polars as pl
import os
import re
import glob
import sys

proj_dir = "/lustre1/project/stg_00090/ASA/"

In [2]:
pd.options.display.max_columns = 250 #Changes the number of columns diplayed (default is 20)
pd.options.display.max_rows = 250 #Changes the number of rows diplayed (default is 60)
pd.options.display.max_colwidth = 500 #Changes the number of characters in a cell so that the contents don't get truncated (default is 50)

# 1. Clean info from ASAP_SAMPLES_DB table (SingleCellAssays)

In [4]:
# Table from Koen - version from 09.04.2024

# in older copies of excel, version = ["v260124", "v160124"], in output tables versions are written as below
versions = ["20240624", "20240527", "20240410", "20240126", "20240116", "20230525"]

current_version = versions[0]

#path = f"/lustre1/project/stg_00090/ASA/doc/ASAP_SAMPLES_DB_31.05_{current_version}.xlsx"
path = f"{proj_dir}/doc/ASAP_SAMPLES_DB_31.05_{current_version}.xlsx"
xls = pd.ExcelFile(path)

### Clean / process the table 

In [5]:
# Getting sheet with Single Cell Assays info,
df = xls.parse("SingleCellAssays", skiprows = 1)

# Rename first column
df.rename(columns = {"Unnamed: 0": "sample_id"}, inplace = True)

# Add leading zeroes to Exp_Assay (8 -> 008 etc); in v120523 need to remove decimals at the end (1.0, 2.0 ...)
#df['Exp_Assay'] = df['Exp_Assay'].apply(lambda x: '{0:0>3}'.format(x)) - worked in v180123
df['Exp_Assay'] = df['Exp_Assay'].astype(str).apply(lambda x: x.replace('.0','')).apply(lambda x: '{0:0>3}'.format(x)) 

# Filtering table (remove bad quality, keep brain ATAC)
df = df.query('goodbad != "bad" & Modality == "ATAC" & sample == "brain"')

# Currently remove MO023
df = df.query('~(Exp_Assay == "023" & Assay == "MO")')

# Generate short barcode ids - will be added to cell barcodes
df["experiment_id"] = df["Assay"] + df["Exp_Assay"].astype(str) 
df["short_bc_id"] = df["Assay"] + df["Exp_Assay"].astype(str) + df["lane"]

# Determine number of donors per sample
df["nSamples"] = df["pool"].str.split("_").apply(lambda x: len(x))

### Get merged samples 
Here we group by experiment and pool - and merge lanes if they correspond to exactly the same pool  
This is important for merging the samples later and modifying barcode names  
If donors in the lanes overlap partially, lanes are not merged  

In [6]:
df_grouped = df.groupby(["experiment_id", "pool"])["lane"].apply(lambda x: "".join(x.unique())).reset_index()
df_grouped.rename(columns = {"lane": "lane_combined"}, inplace = True)

# merged sample names - for merging multiple runs for the same sample
df_grouped["merged_sample_name"] = df_grouped["experiment_id"] + df_grouped["lane_combined"] 

df_grouped.head()

,experiment_id,pool,lane_combined,merged_sample_name
0,AT001,76b_77b_78b_79b_80b_31b_32b_33b_34b_35b,abc,AT001abc
1,AT002,6b_12b_13b_27b_28b_37b_38b_39b_40b_44b,ab,AT002ab
2,AT003,101b_102b_103b_104b_105b_106b_107b_108b_109b_110b,abcd,AT003abcd
3,AT004,2a_3a_4a_5a_6a_12a_13a_14a_15a_36a_26a_27a_28a_29a_30a_31a_32a_33a_34a_35a,abcd,AT004abcd
4,AT005,121b_122b_123b_124b_125b_126b_127b_128b_129b_130b_161b_162b_163b_164b_165b_166b_167b_168b_169b_170b,abcd,AT005abcd


In [7]:
# add combined lane info to initial table, get subset of columns 

df = pd.merge(df, df_grouped[["experiment_id", "pool", "merged_sample_name"]], on = ["experiment_id", "pool"], how = "inner")
df = df[["sample_id", "Assay", "Exp_Assay", "lane", "experiment_id", "pool", "nSamples", "short_bc_id", "merged_sample_name"]]

df.head()

,sample_id,Assay,Exp_Assay,lane,experiment_id,pool,nSamples,short_bc_id,merged_sample_name
0,20210527_10xMO-atac_asaEZ,MO,001,a,MO001,14b_15b,2,MO001a,MO001abc
1,20210527_10xMO-atac_asaNP,MO,001,b,MO001,14b_15b,2,MO001b,MO001abc
2,20210527_10xMO-atac_asa10x,MO,001,c,MO001,14b_15b,2,MO001c,MO001abc
3,20210616_10xMO-ATAC-NP05-SNa,MO,002,a,MO002,7a_8a_9a_10a_11a,5,MO002a,MO002ac
4,20210616_10xMO-ATAC-NP05-SNb,MO,002,c,MO002,7a_8a_9a_10a_11a,5,MO002c,MO002ac


## 2. Get seqeuncing info  

File with names of sequencing runs is generated with code from Gert   
Command to find all samples for a sequening project (sample name, sequencing run and technique): 


conda_initialize  /lustre1/project/stg_00002/mambaforge/vsc30366  
conda activate /lustre1/project/stg_00002/mambaforge/vsc30366/envs/ngs_tools

current_version=20240624

/staging/leuven/stg_00002/lcb/ngs_runs/ngs_tools/lcb_ngs_metadata.py project ASA | sort > /staging/leuven/stg_00090/ASA/doc/sequencing_runs_metadata_${current_version}.tsv 


In [8]:
# Parse sequencing runs info
#path = "/staging/leuven/stg_00090/ASA/doc/sequencing_runs_metadata_25052023.tsv"

print(current_version)
path = f"/staging/leuven/stg_00090/ASA/doc/sequencing_runs_metadata_{current_version}.tsv"
#path = f"/data/leuven/303/vsc30366/jupyter_notebooks/ASA/sequencing_runs_metadata_{current_version}.tsv"

df_sequencing = pd.read_csv(path, sep = "\t")
df_sequencing.rename(columns = {"# sample_name": "sample_id", "samplesheet_file_name": "sample_name", "technique": "technology" }, inplace = True)

# NovaSeqX_20240430 is called NovaSeqXa_20240430 (as there were 2 runs on that day).
df_sequencing["sequencing_run_name"][df_sequencing["sequencing_run_name"] == "NovaSeqX_20240430"] = "NovaSeqXa_20240430"

df_sequencing["sample_name_full"] = df_sequencing["sequencing_run_name"] + "/" + df_sequencing["sample_name"] 


# Add seqeuncing info to the table
df = pd.merge(df, df_sequencing, on = "sample_id", how = "inner")

df.head()

20240624


,sample_id,Assay,Exp_Assay,lane,experiment_id,pool,nSamples,short_bc_id,merged_sample_name,sample_name,sequencing_run_name,technology,sample_name_full
0,20210527_10xMO-atac_asaEZ,MO,001,a,MO001,14b_15b,2,MO001a,MO001abc,ASA__811eca__20210527_10xMO-atac_asaEZ,NextSeq2000_20210603,10x Multiome ATAC,NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ
1,20210527_10xMO-atac_asaNP,MO,001,b,MO001,14b_15b,2,MO001b,MO001abc,ASA__f63e18__20210527_10xMO-atac_asaNP,NextSeq2000_20210603,10x Multiome ATAC,NextSeq2000_20210603/ASA__f63e18__20210527_10xMO-atac_asaNP
2,20210527_10xMO-atac_asa10x,MO,001,c,MO001,14b_15b,2,MO001c,MO001abc,ASA__e77d5a__20210527_10xMO-atac_asa10x,NextSeq2000_20210603,10x Multiome ATAC,NextSeq2000_20210603/ASA__e77d5a__20210527_10xMO-atac_asa10x
3,20210616_10xMO-ATAC-NP05-SNa,MO,002,a,MO002,7a_8a_9a_10a_11a,5,MO002a,MO002ac,ASA__69a0dd__20210616_10xMO-ATAC-NP05-SNa,NextSeq2000_20210624,10x Multiome ATAC,NextSeq2000_20210624/ASA__69a0dd__20210616_10xMO-ATAC-NP05-SNa
4,20210616_10xMO-ATAC-NP05-SNa,MO,002,a,MO002,7a_8a_9a_10a_11a,5,MO002a,MO002ac,ASA__69a0dd__20210616_10xMO-ATAC-NP05-SNa,NovaSeq6000_20211025,10x Multiome ATAC,NovaSeq6000_20211025/ASA__69a0dd__20210616_10xMO-ATAC-NP05-SNa


### Finalizing formating of 'technology' 

Field 'technology' should then match VSN config with correct list of barcodes  
It is important to remove '10x' from the beginning -> otherwise gives error in VSN

set technology name for scale ATAC:   
only 211202_ScaleBio-001-ATAC is with the commercial kit.  
211202_ScaleIH-002-ATAC = inhouse but short barcode  
AT-003, 4, 5, 6, 7 etc are inhouse design but same lenght of barcode as the commercial kit"  

In [9]:
print(set(df.technology.tolist()))

{'HyDrop-scaleATAC 96 ligation', 'HyDrop-ATAC 96 ligation', '10x scATAC v2', '10x scATAC v1.1', '10x scATAC', 'ScaleATAC', 'scaleATAC', '10x Multiome ATAC'}


In [10]:
df["techology_original"] = df.technology

df.technology.replace(["10x", " ", "v1.1", "v2"], "", regex=True, inplace = True)

df.loc[df.technology == "ScaleATAC", "technology"] = "ScaleBioIH"
df.loc[df.technology == "scaleATAC", "technology"] = "ScaleBioIH"
df.loc[df.sample_id == "211202_ScaleIH-002-ATAC", "technology"] = "ScaleBioIH6"
df.loc[df.sample_id == "211202_ScaleBio-001-ATAC", "technology"] = "ScaleBio"
df.loc[df.technology == "HyDrop-ATAC96ligation", "technology"] = "HYAv2"
df.loc[df.technology == "HyDrop-scaleATAC96ligation", "technology"] = "ScaleBioIH-HYAv2"

df.technology.value_counts()


MultiomeATAC        118
scATAC               88
ScaleBioIH           63
HYAv2                11
ScaleBioIH-HYAv2      2
ScaleBio              1
ScaleBioIH6           1
Name: technology, dtype: int64

### Final table per sequencing run - overview

sample_id: ID from Koen's table;  
sample_name - base for filenames in fastq directory  
sample_name_full - sequencing_run_name/sample_name (directory for fastq file - each contains multiple files after demultiplexing)

In [11]:
df.head()

,sample_id,Assay,Exp_Assay,lane,experiment_id,pool,nSamples,short_bc_id,merged_sample_name,sample_name,sequencing_run_name,technology,sample_name_full,techology_original
0,20210527_10xMO-atac_asaEZ,MO,001,a,MO001,14b_15b,2,MO001a,MO001abc,ASA__811eca__20210527_10xMO-atac_asaEZ,NextSeq2000_20210603,MultiomeATAC,NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ,10x Multiome ATAC
1,20210527_10xMO-atac_asaNP,MO,001,b,MO001,14b_15b,2,MO001b,MO001abc,ASA__f63e18__20210527_10xMO-atac_asaNP,NextSeq2000_20210603,MultiomeATAC,NextSeq2000_20210603/ASA__f63e18__20210527_10xMO-atac_asaNP,10x Multiome ATAC
2,20210527_10xMO-atac_asa10x,MO,001,c,MO001,14b_15b,2,MO001c,MO001abc,ASA__e77d5a__20210527_10xMO-atac_asa10x,NextSeq2000_20210603,MultiomeATAC,NextSeq2000_20210603/ASA__e77d5a__20210527_10xMO-atac_asa10x,10x Multiome ATAC
3,20210616_10xMO-ATAC-NP05-SNa,MO,002,a,MO002,7a_8a_9a_10a_11a,5,MO002a,MO002ac,ASA__69a0dd__20210616_10xMO-ATAC-NP05-SNa,NextSeq2000_20210624,MultiomeATAC,NextSeq2000_20210624/ASA__69a0dd__20210616_10xMO-ATAC-NP05-SNa,10x Multiome ATAC
4,20210616_10xMO-ATAC-NP05-SNa,MO,002,a,MO002,7a_8a_9a_10a_11a,5,MO002a,MO002ac,ASA__69a0dd__20210616_10xMO-ATAC-NP05-SNa,NovaSeq6000_20211025,MultiomeATAC,NovaSeq6000_20211025/ASA__69a0dd__20210616_10xMO-ATAC-NP05-SNa,10x Multiome ATAC


In [12]:
df["sequencing_run_name"].unique()

array(['NextSeq2000_20210603', 'NextSeq2000_20210624',
       'NovaSeq6000_20211025', 'NextSeq2000_20210720',
       'NovaSeq6000_20210927', 'NextSeq2000_20220104',
       'NextSeq2000_20211214', 'NextSeq2000_20220217',
       'NextSeq2000_20220207', 'NextSeq2000_20220223',
       'NextSeq2000_20220324', 'NextSeq2000_20220630',
       'NovaSeq6000_20221107', 'NextSeq2000_20220509',
       'NextSeq2000_20220623', 'NextSeq2000_20220624',
       'NextSeq2000_20230306', 'NextSeq2000_20231031',
       'NextSeq2000_20220706', 'NovaSeq6000_20220824',
       'NextSeq2000_20220711', 'NextSeq2000_20220722',
       'NextSeq2000_20220801', 'NextSeq2000_20231108',
       'NextSeq2000_20220825', 'NovaSeq6000_20220909',
       'NovaSeq6000_20231206', 'NextSeq2000_20220826',
       'NextSeq2000_20221004', 'NextSeq2000_20221222',
       'NextSeq2000_20230110', 'NextSeq2000_20230214',
       'NovaSeq6000_20240116', 'NextSeq2000_20230316',
       'NextSeq2000_20230403', 'NovaSeq6000_20230503',
       'Ne

# 3. Add paths to fastq files

In [15]:
# now add paths to R1-R3 directly from fastq folder
fastq_dir = proj_dir + "fastq"

# using here full sample names that have format "sequencing_run_name/sample_name" (sequencing_run_name are subfolders with fastq files for each sample)
sample_names_full = set(df.sample_name_full.tolist())

fastq_df_merged = pd.DataFrame(columns=["sample_name_full", "fastq_PE1_path", "fastq_barcode_path", "fastq_PE2_path",])


for sample in sample_names_full:
    

    fastq_list_r1 = sorted(
        glob.glob(
            f"{fastq_dir}/{sample}*R1*.fastq.gz"
        )
    )
    fastq_list_r2 = sorted(
        glob.glob(
            f"{fastq_dir}/{sample}*R2*.fastq.gz"
        )
    )
    fastq_list_r3 = sorted(
        glob.glob(
            f"{fastq_dir}/{sample}*R3*.fastq.gz"
        )
    )
    
    if not len(fastq_list_r1) == len(fastq_list_r2) == len(fastq_list_r3):
        print(sample)
        print(f"\t{len(fastq_list_r1)}, {len(fastq_list_r2)}, {len(fastq_list_r3)}")
        sys.exit('R1-R3 files dont match in number')
        
    if len(fastq_list_r1) == 0:
        print(f"sample {sample}: fastq files missing - {fastq_dir}/{sample}*R1*.fastq.gz!")
        
    
    # df for each sample
    fastq_df = pd.DataFrame(columns=["fastq_PE1_path", "fastq_barcode_path", "fastq_PE2_path"])
    fastq_df["fastq_PE1_path"] = fastq_list_r1
    fastq_df["fastq_barcode_path"] = fastq_list_r2
    fastq_df["fastq_PE2_path"] = fastq_list_r3
    fastq_df["sample_name_full"] = sample
    
    # combining for all samples
    fastq_df_merged = pd.concat([fastq_df_merged, fastq_df])
    

    
fastq_df_merged = fastq_df_merged.reset_index(drop=True)

Now combine metadata with paths to fastq - to get final metadata table that can be used in VSN pipeline  
This file will have more lines becasue there are multiple fastq files per seqeuncing run (after demultiplexing), which all will be merged

In [17]:
df_extended_fastq = pd.merge(df, fastq_df_merged, on = "sample_name_full", how = "inner")

# check that all fastq files are uniquily connected to samples (same number of rows in fastq_df_merged and df4) 
print(df.shape)
print(fastq_df_merged.shape)
print(df_extended_fastq.shape) 

# This is the final metadata table that will be used in the downstream analysis
#outf = f"{proj_dir}/doc/metadata_atac_vsn_preprocess_{current_version}.tsv"
outf = f"{proj_dir}/doc/metadata_atac_vsn_preprocess_{current_version}.tsv"
print(outf)

(284, 14)
(2431, 4)
(2431, 17)
/lustre1/project/stg_00090/ASA//doc/metadata_atac_vsn_preprocess_20240624.tsv


In [18]:
df_extended_fastq.to_csv(outf, index = False, sep = "\t")

# Control: compare which samples are added compared to earlier version

In [25]:
# new list of sequencing runs
df_extended_fastq.head()

,sample_id,Assay,Exp_Assay,lane,experiment_id,pool,nSamples,short_bc_id,merged_sample_name,sample_name,sequencing_run_name,technology,sample_name_full,techology_original,fastq_PE1_path,fastq_barcode_path,fastq_PE2_path
0,20210527_10xMO-atac_asaEZ,MO,001,a,MO001,14b_15b,2,MO001a,MO001abc,ASA__811eca__20210527_10xMO-atac_asaEZ,NextSeq2000_20210603,MultiomeATAC,NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ,10x Multiome ATAC,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S1_L001_R1_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S1_L001_R2_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S1_L001_R3_001.fastq.gz
1,20210527_10xMO-atac_asaEZ,MO,001,a,MO001,14b_15b,2,MO001a,MO001abc,ASA__811eca__20210527_10xMO-atac_asaEZ,NextSeq2000_20210603,MultiomeATAC,NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ,10x Multiome ATAC,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S1_L002_R1_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S1_L002_R2_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S1_L002_R3_001.fastq.gz
2,20210527_10xMO-atac_asaEZ,MO,001,a,MO001,14b_15b,2,MO001a,MO001abc,ASA__811eca__20210527_10xMO-atac_asaEZ,NextSeq2000_20210603,MultiomeATAC,NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ,10x Multiome ATAC,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S2_L001_R1_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S2_L001_R2_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S2_L001_R3_001.fastq.gz
3,20210527_10xMO-atac_asaEZ,MO,001,a,MO001,14b_15b,2,MO001a,MO001abc,ASA__811eca__20210527_10xMO-atac_asaEZ,NextSeq2000_20210603,MultiomeATAC,NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ,10x Multiome ATAC,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S2_L002_R1_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S2_L002_R2_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S2_L002_R3_001.fastq.gz
4,20210527_10xMO-atac_asaEZ,MO,001,a,MO001,14b_15b,2,MO001a,MO001abc,ASA__811eca__20210527_10xMO-atac_asaEZ,NextSeq2000_20210603,MultiomeATAC,NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ,10x Multiome ATAC,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S3_L001_R1_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S3_L001_R2_001.fastq.gz,/lustre1/project/stg_00090/ASA/fastq/NextSeq2000_20210603/ASA__811eca__20210527_10xMO-atac_asaEZ_S3_L001_R3_001.fastq.gz


In [24]:
runs_new = df_extended_fastq["sample_name_full"].tolist()
samples_new = df_extended_fastq["short_bc_id"].tolist()

versions = ["20240624", "20240527", "20240410", "20240126", "20240116", "20230525"]
versions = ["20240624", "20240527"]


for old_version in versions[1:len(versions)]:
    
    df_old = pd.read_csv(f"{proj_dir}/doc/metadata_atac_vsn_preprocess_{old_version}.tsv", sep = "\t")
    runs_old = df_old["sample_name_full"].tolist()
    samples_old = df_old["short_bc_id"].tolist()
    
    print(f"Comparing with version from {old_version}:")
    new_seqeucning_runs = [x for x in runs_new if x not in runs_old]
    new_samples = [x for x in samples_new if x not in samples_old]
    
    print(f"  - Number of new sequencing runs: {len(new_seqeucning_runs)}") 
    print(new_seqeucning_runs)
    
    print(f"  - New samples ({len(set(new_samples))}): {set(new_samples)}") 
    print(set(new_samples))
    


Comparing with version from 20240527:
  - Number of new sequencing runs: 82
['NovaSeqXa_20240611/ASA__5f5803__20240424_AT-026-a-ATAC', 'NovaSeqXa_20240611/ASA__5f5803__20240424_AT-026-a-ATAC', 'NovaSeqXa_20240611/ASA__5f5803__20240424_AT-026-a-ATAC', 'NovaSeqXa_20240611/ASA__5f5803__20240424_AT-026-a-ATAC', 'NovaSeqXa_20240611/ASA__5f5803__20240424_AT-026-a-ATAC', 'NovaSeqXa_20240611/ASA__5f5803__20240424_AT-026-a-ATAC', 'NovaSeqXa_20240611/ASA__5f5803__20240424_AT-026-a-ATAC', 'NovaSeqXa_20240611/ASA__5f5803__20240424_AT-026-a-ATAC', 'NovaSeqXa_20240611/ASA__3338d0__20240424_AT-026-e-ATAC', 'NovaSeqXa_20240611/ASA__3338d0__20240424_AT-026-e-ATAC', 'NovaSeqXa_20240611/ASA__3338d0__20240424_AT-026-e-ATAC', 'NovaSeqXa_20240611/ASA__3338d0__20240424_AT-026-e-ATAC', 'NovaSeqXa_20240611/ASA__3338d0__20240424_AT-026-e-ATAC', 'NovaSeqXa_20240611/ASA__3338d0__20240424_AT-026-e-ATAC', 'NovaSeqXa_20240611/ASA__3338d0__20240424_AT-026-e-ATAC', 'NovaSeqXa_20240611/ASA__3338d0__20240424_AT-026-e-AT

# 4. Add samples metadata - add brain region and CN/PD info (Sample_overview)

In [26]:
def match_brain_region(abr):

    if abr.endswith("a"):
        region = "substantia_nigra"
    elif abr.endswith("b"):
        region = "cyngulate_cortex"
    elif abr.endswith("d"):
        region = "motor_cortex"
    else:
        raise ValueError(f"Abbreviation {abr} doesn't match any brain region!")

    return region

In [27]:
# Retrieve donor information
samples_overview = xls.parse("Sample_overview", skiprows = 1)
donor_info = pl.DataFrame(samples_overview[['CN/PD', 'patient_nr']]).unique()
donor_info.head()

CN/PD,patient_nr
str,i64
"""CN""",1
"""CN""",2
"""CN""",3
"""CN""",4
"""CN""",5


In [28]:
# subset the columns for samples metadata 
metadata = pl.from_pandas(df_extended_fastq).select(["merged_sample_name", "short_bc_id", "pool", "nSamples", "technology"]).unique()

# split donors (keep info on the pool)
metadata_long  = (metadata.
    with_column(pl.col("pool").str.split("_").alias("donor_id")).
    explode("donor_id"))

# define brain region based on a/b
metadata_long = (metadata_long.
                with_columns([pl.col("donor_id").
                            apply(lambda x: match_brain_region(x)).
                            alias("brain_region"),
                            pl.col("donor_id").
                            apply(lambda x: int(re.sub("[abd]", "", x))).
                            alias("patient_nr")]))
print(metadata_long.shape)


# add CN/PD info
metadata_long = metadata_long.join(donor_info, on = "patient_nr")

# aggregate back by short_bc_id
metadata_grouped = (metadata_long.
     groupby(["merged_sample_name", "short_bc_id", "pool", "nSamples", "technology"]).
     agg([pl.col("brain_region").unique().sort().apply(lambda x: ".".join(x)).alias("brain_region"),
          pl.col("CN/PD").unique().sort().apply(lambda x: ".".join(x)).alias("donor_type")
          ]).
     sort(["brain_region", "short_bc_id"]))

print(metadata_grouped.shape) 
metadata_grouped.head()

(1567, 8)
(167, 7)


merged_sample_name,short_bc_id,pool,nSamples,technology,brain_region,donor_type
str,str,str,i64,str,str,str
"""AT001abc""","""AT001a""","""76b_77b_78b_79...",10,"""scATAC""","""cyngulate_cort...","""CN.PD"""
"""AT001abc""","""AT001b""","""76b_77b_78b_79...",10,"""scATAC""","""cyngulate_cort...","""CN.PD"""
"""AT001abc""","""AT001c""","""76b_77b_78b_79...",10,"""scATAC""","""cyngulate_cort...","""CN.PD"""
"""AT002ab""","""AT002a""","""6b_12b_13b_27b...",10,"""ScaleBio""","""cyngulate_cort...","""CN"""
"""AT002ab""","""AT002b""","""6b_12b_13b_27b...",10,"""ScaleBioIH6""","""cyngulate_cort...","""CN"""


In [29]:
metadata_grouped.to_pandas().brain_region.value_counts()

cyngulate_cortex                     100
substantia_nigra                      56
cyngulate_cortex.substantia_nigra      9
motor_cortex                           2
Name: brain_region, dtype: int64

In [30]:
outf = f"{proj_dir}/doc/atac_samples_metadata_with_donor_region_info_{current_version}.csv"
outf

'/lustre1/project/stg_00090/ASA//doc/atac_samples_metadata_with_donor_region_info_20240624.csv'

In [31]:
metadata_grouped.write_csv(outf)

In [32]:
# control CC samples, MO
metadata_grouped.filter((pl.col("brain_region") == "cyngulate_cortex") & (pl.col("donor_type") == "CN") & (pl.col("technology") == "MultiomeATAC" ))

merged_sample_name,short_bc_id,pool,nSamples,technology,brain_region,donor_type
str,str,str,i64,str,str,str
"""MO001abc""","""MO001a""","""14b_15b""",2,"""MultiomeATAC""","""cyngulate_cort...","""CN"""
"""MO001abc""","""MO001b""","""14b_15b""",2,"""MultiomeATAC""","""cyngulate_cort...","""CN"""
"""MO001abc""","""MO001c""","""14b_15b""",2,"""MultiomeATAC""","""cyngulate_cort...","""CN"""
"""MO002bd""","""MO002b""","""7b_8b_9b_10b_1...",5,"""MultiomeATAC""","""cyngulate_cort...","""CN"""
"""MO002bd""","""MO002d""","""7b_8b_9b_10b_1...",5,"""MultiomeATAC""","""cyngulate_cort...","""CN"""
"""MO004a""","""MO004a""","""16b_17b_18b_19...",5,"""MultiomeATAC""","""cyngulate_cort...","""CN"""
"""MO007a""","""MO007a""","""31b_32b_33b_34...",5,"""MultiomeATAC""","""cyngulate_cort...","""CN"""
"""MO008a""","""MO008a""","""1b_2b_3b_36b_3...",5,"""MultiomeATAC""","""cyngulate_cort...","""CN"""
"""MO010ab""","""MO010a""","""45b_46b_72b_74...",4,"""MultiomeATAC""","""cyngulate_cort...","""CN"""


In [33]:
# motor cortex sample (new)
metadata_grouped.filter((pl.col("brain_region") == "motor_cortex"))

merged_sample_name,short_bc_id,pool,nSamples,technology,brain_region,donor_type
str,str,str,i64,str,str,str
"""MO019bd""","""MO019b""","""119d_122d_126d...",5,"""MultiomeATAC""","""motor_cortex""","""CN"""
"""MO019bd""","""MO019d""","""119d_122d_126d...",5,"""MultiomeATAC""","""motor_cortex""","""CN"""
